# PCA, then clustering

Does reducing the dimensionality help or hurt?

We keep only the first *d* principal components and re-run the three clustering methods
from `04_Compare.ipynb` on those.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from scipy.cluster.hierarchy import linkage, fcluster

from pam import pam, labels as pam_labels

DATA = "../week_2/data"

d = np.load(f"{DATA}/embeddings_stl10_1000.npz")

X = d["E_l12"]
y = d["y"]

K = 10
RANDOM_STATE = 0
DIMS = [2, 5, 10, 32, 50, 100, 768]

print(X.shape)

In [ ]:
rows = []

for dim in DIMS:
    Xd = X if dim == X.shape[1] else PCA(n_components=dim, random_state=0).fit_transform(X)

    # --- k-means -------------------------------------------------
    t0 = time.perf_counter()
    km = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(Xd)
    rows.append({
        "components": dim, "method": "k-means",
        "ARI vs truth": adjusted_rand_score(y, km.labels_),
        "NMI": normalized_mutual_info_score(y, km.labels_),
        "seconds": time.perf_counter() - t0,
    })

    # --- k-medoids -----------------------------------------------
    t0 = time.perf_counter()
    medoids, _ = pam(Xd, K, random_state=RANDOM_STATE)
    rows.append({
        "components": dim, "method": "k-medoids",
        "ARI vs truth": adjusted_rand_score(y, pam_labels(Xd, medoids)),
        "NMI": normalized_mutual_info_score(y, pam_labels(Xd, medoids)),
        "seconds": time.perf_counter() - t0,
    })

    # --- hierarchical --------------------------------------------
    t0 = time.perf_counter()
    lb = fcluster(linkage(Xd, method="ward"), t=K, criterion="maxclust")
    rows.append({
        "components": dim, "method": "hierarchical",
        "ARI vs truth": adjusted_rand_score(y, lb),
        "NMI": normalized_mutual_info_score(y, lb),
        "seconds": time.perf_counter() - t0,
    })

results = (pd.DataFrame(rows)
           .sort_values(["components", "method"])
           .reset_index(drop=True))

In [ ]:
results.pivot(index="components", columns="method",
               values="ARI vs truth").style.format("{:.3f}")

### ARI against the number of components

In [ ]:
wide = results.pivot(index="components", columns="method", values="ARI vs truth")

fig, ax = plt.subplots(figsize=(7, 3.6))

for method in wide.columns:
    ax.plot(wide.index, wide[method], marker="o", label=method)

ax.axhline(0.1, color="grey", ls="--", lw=1, label="chance")
ax.set_xscale("log")
ax.set_xlabel("components kept")
ax.set_ylabel("ARI vs truth")
ax.legend()

plt.tight_layout()
plt.show()

### Results